# Untouched final evaluation and fail-closed release

This is the only v2 notebook allowed to load `20259` or `20265`. Run it once after Notebooks 2 and 3 are locked. Failed gates produce a diagnostic report and do not overwrite the production model.

In [ ]:
from __future__ import annotations
import hashlib, inspect, json, os, platform, sys
from pathlib import Path
import joblib, numpy as np, pandas as pd, sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name == 'v2' or not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'v2'
CACHE_ROOT = ARTIFACT_ROOT / 'cache'
MODEL_ROOT = PROJECT_ROOT / 'model'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 20260812
SESSION_ORDER = ['20229','20235','20239','20245','20249','20255','20259','20265']
SEASONS = {'fall_winter':['20229','20239','20249','20259'], 'summer':['20235','20245','20255','20265']}
FINAL_TEST = {'fall_winter':'20259', 'summer':'20265'}
DEVELOPMENT = {k:[s for s in v if s != FINAL_TEST[k]] for k,v in SEASONS.items()}

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

def fingerprint(payload) -> str:
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]

def versions():
    return {'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
            'scikit_learn':sklearn.__version__,'joblib':joblib.__version__}
BASE = ['position_to_capacity','waitlist_to_capacity','days_to_deadline','movement_3d','movement_7d','position','waitlist','capacity','capacity_changed_7d','position_to_waitlist','days_squared','log_waitlist','movement_velocity_7d']
CONTEXT = BASE + ['near_deadline_7d','days_under_7','days_under_14','days_over_60','position_ratio_near_7d','waitlist_ratio_near_7d','rank_over_30pct','campus_erin','campus_scar','term_winter','term_full_year','winter_near_7d','scar_near_7d']
RANK_MONOTONIC={'position_to_capacity':-1,'position':-1,'position_to_waitlist':-1,'position_ratio_near_7d':-1,'rank_over_30pct':-1}
def targeted(frame):
    f=frame.copy(); days=f.days_to_deadline.astype(float); near=(days<=7).astype('float32')
    f['near_deadline_7d']=near; f['days_under_7']=(7-days).clip(lower=0); f['days_under_14']=(14-days).clip(lower=0); f['days_over_60']=(days-60).clip(lower=0)
    f['position_ratio_near_7d']=f.position_to_capacity*near; f['waitlist_ratio_near_7d']=f.waitlist_to_capacity*near
    f['rank_over_30pct']=(f.position_to_capacity>.30).astype('float32'); f['campus_erin']=(f.campus=='ERIN').astype('float32'); f['campus_scar']=(f.campus=='SCAR').astype('float32')
    f['term_winter']=(f.term=='winter').astype('float32'); f['term_full_year']=(f.term=='full_year').astype('float32')
    f['winter_near_7d']=f.term_winter*near; f['scar_near_7d']=f.campus_scar*near
    return f
def make_model(features, *, leaf=15, l2=3.0):
    constraints=[RANK_MONOTONIC.get(x,0) for x in features]
    return Pipeline([('features',ColumnTransformer([('numeric',SimpleImputer(strategy='median'),features)],remainder='drop')),
      ('model',HistGradientBoostingClassifier(max_iter=200,learning_rate=.05,max_leaf_nodes=leaf,l2_regularization=l2,monotonic_cst=constraints,random_state=RANDOM_STATE))])
def ece(y,p,w,bins=10):
    edges=np.linspace(0,1,bins+1); ids=np.clip(np.digitize(p,edges)-1,0,bins-1); total=w.sum(); out=0
    for b in range(bins):
        m=ids==b
        if m.any(): out+=w[m].sum()/total*abs(np.average(y[m],weights=w[m])-np.average(p[m],weights=w[m]))
    return float(out)
def metrics(frame,p):
    y=frame.cleared.to_numpy(); w=frame.model_weight.to_numpy(); p=np.clip(np.asarray(p),1e-6,1-1e-6)
    return {'brier':brier_score_loss(y,p,sample_weight=w),'log_loss':log_loss(y,p,sample_weight=w,labels=[0,1]),'ece':ece(y,p,w),'auc':roc_auc_score(y,p,sample_weight=w),'accuracy':np.average((p>=.5)==y,weights=w)}
def export_tree(predictor):
    return [{'value':float(n['value']),'feature':int(n['feature_idx']),'threshold':float(n['num_threshold']),
      'missing_left':bool(n['missing_go_to_left']),'left':int(n['left']),'right':int(n['right']),'leaf':bool(n['is_leaf'])} for n in predictor.nodes]
def export_model(pipeline,features):
    estimator=pipeline.named_steps['model']; imputer=pipeline.named_steps['features'].named_transformers_['numeric']
    return {'numeric_features':features,'imputation_values':[float(x) for x in imputer.statistics_],
      'baseline_log_odds':float(np.asarray(estimator._baseline_prediction).ravel()[0]),
      'trees':[export_tree(stage[0]) for stage in estimator._predictors],'calibration':{'method':'none'}}


## Load locked specification and untouched sessions

In [ ]:
paths=sorted(ARTIFACT_ROOT.glob('locked-spec-*.json'))
if len(paths)!=1: raise RuntimeError(f'Expected exactly one locked spec, found {len(paths)}')
locked=json.loads(paths[0].read_text()); spec=locked['spec']; features=spec['features']
cache_manifest=json.loads((ARTIFACT_ROOT/'cache-manifest.json').read_text())
MAX_NEGATIVE_CENSOR_GAP_HOURS=cache_manifest['negative_censor_gap_hours_max']
samples={s:targeted(pd.read_pickle(CACHE_ROOT/f'{s}-positions-v2.pkl')) for s in SESSION_ORDER}
daily={s:pd.read_pickle(CACHE_ROOT/f'{s}-daily-v2.pkl') for s in SESSION_ORDER}


## Complete-trajectory historical baseline and simple rules

In [ ]:
def historical_probability(train_sessions,valid):
    result=np.full(len(valid),np.nan)
    histories=pd.concat([daily[s] for s in train_sessions],ignore_index=True)
    for i,(_,row) in enumerate(valid.iterrows()):
        course=histories.loc[histories.course_code.eq(row.course_code)]
        outcomes=[]
        for _,offering in course.groupby('offering_id',sort=False):
            # Short logs were truncated during reconstruction; no missing value is ever replaced by zero.
            nearest=offering.iloc[(offering.days_to_deadline-row.days_to_deadline).abs().argsort()[:1]]
            if nearest.empty or int(nearest.iloc[0].waitlist)<int(row.position): continue
            cleared=nearest.iloc[0].net_drop_to_deadline>=row.position
            if not cleared and nearest.iloc[0].terminal_gap_hours>MAX_NEGATIVE_CENSOR_GAP_HOURS: continue
            outcomes.append(int(cleared))
        if outcomes: result[i]=np.mean(outcomes)
    return result
def apply_locked_calibration(season,p):
    if locked['calibration'][season]=='none': return np.asarray(p)
    params=locked['calibration_parameters'][season]; clipped=np.clip(np.asarray(p),1e-6,1-1e-6)
    score=params['coefficient']*np.log(clipped/(1-clipped))+params['intercept']
    return 1/(1+np.exp(-score))


## One-time final evaluation with clustered uncertainty

In [ ]:
def cluster_ci(frame,p,reps=2000):
    rng=np.random.default_rng(RANDOM_STATE); ids=frame.offering_id.unique(); loc={x:np.flatnonzero(frame.offering_id.to_numpy()==x) for x in ids}; vals=[]
    for _ in range(reps):
        idx=np.concatenate([loc[x] for x in rng.choice(ids,len(ids),replace=True)]); vals.append(metrics(frame.iloc[idx],np.asarray(p)[idx])['brier'])
    return np.quantile(vals,[.025,.975]).tolist()
rows=[]; fitted={}
for season,test_session in FINAL_TEST.items():
    train_sessions=DEVELOPMENT[season]; train=pd.concat([samples[s] for s in train_sessions],ignore_index=True); test=samples[test_session]
    model=make_model(**spec); model.fit(train[features],train.cleared,model__sample_weight=train.model_weight); fitted[season]=model
    oracle=apply_locked_calibration(season,model.predict_proba(test[features])[:,1]); literal=(test.position_to_capacity<=.10).astype(float).to_numpy(); history=historical_probability(train_sessions,test)
    for name,p,mask in [('oracle',oracle,np.ones(len(test),bool)),('literal_10_percent',literal,np.ones(len(test),bool)),('historical_percentage',history,np.isfinite(history))]:
        scored=test.loc[mask]; pred=np.asarray(p)[mask]
        rows.append({'season':season,'method':name,'coverage':float(scored.model_weight.sum()/test.model_weight.sum()),**metrics(scored,pred),'brier_ci':cluster_ci(scored,pred)})
final_scores=pd.DataFrame(rows); final_scores

## Release gates

In [ ]:
THRESHOLDS={'oracle_brier_lt_literal':True,'ece_lte':.05,'near_deadline_gap_lte':.08,'large_queue_gap_lte':.08,'max_probability_gap_lte':.10}
def gap(frame,p,mask):
    if not mask.any() or mask.all(): return np.nan
    return abs(np.average(frame.cleared[mask],weights=frame.model_weight[mask])-np.average(np.asarray(p)[mask],weights=frame.model_weight[mask]))
gate_rows=[]
for season,test_session in FINAL_TEST.items():
    test=samples[test_session]; model=fitted[season]; p=apply_locked_calibration(season,model.predict_proba(test[features])[:,1])
    oracle=final_scores.query("season==@season and method=='oracle'").iloc[0]; literal=final_scores.query("season==@season and method=='literal_10_percent'").iloc[0]
    bins=pd.cut(p,np.linspace(0,1,11),include_lowest=True); probability_gap=max(abs(np.average(g.cleared,weights=g.model_weight)-np.average(g['_p'],weights=g.model_weight)) for _,g in test.assign(_p=p).groupby(bins,observed=True) if len(g))
    checks={'oracle_brier_lt_literal':oracle.brier<literal.brier,'ece':oracle.ece<=THRESHOLDS['ece_lte'],
      'near_deadline_gap':gap(test,p,test.days_to_deadline.le(7).to_numpy())<=THRESHOLDS['near_deadline_gap_lte'],
      'large_queue_gap':gap(test,p,test.waitlist.ge(100).to_numpy())<=THRESHOLDS['large_queue_gap_lte'],
      'max_probability_gap':probability_gap<=THRESHOLDS['max_probability_gap_lte']}
    gate_rows.append({'season':season,'passed':all(checks.values()),'checks':checks})
gates=pd.DataFrame(gate_rows); gates

## Refit and export only after every gate passes

In [ ]:
report={'locked_spec_fingerprint':locked['fingerprint'],'final_test':FINAL_TEST,'scores':final_scores.to_dict('records'),'gates':gate_rows,'thresholds':THRESHOLDS,'versions':versions(),
 'cache_manifest_hash':sha256(ARTIFACT_ROOT/'cache-manifest.json')}
(ARTIFACT_ROOT/'final-evaluation.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
if not gates.passed.all():
    raise RuntimeError('Release blocked. Diagnostic report written; production model was not overwritten.')
models={}
for season,sessions in SEASONS.items():
    training=pd.concat([samples[s] for s in sessions],ignore_index=True); model=make_model(**spec)
    model.fit(training[features],training.cleared,model__sample_weight=training.model_weight)
    exported=export_model(model,features)
    if locked['calibration'][season]!='none': exported['calibration']={'method':'platt','parameters':locked['calibration_parameters'][season]}
    models[season]={'quality':'validated','training_sessions':sessions,'final_evaluation':next(x for x in report['scores'] if x['season']==season and x['method']=='oracle'),**exported}
artifact={'schema_version':7,'model_type':'seasonal_hist_gradient_boosting_bundle','session_routing':{'5':'summer','9':'fall_winter'},
 'training_support':{'max_position':int(max(f.position.max() for f in samples.values())),'position_never_exceeds_waitlist':True},
 'provenance':{'locked_spec_fingerprint':locked['fingerprint'],'cache_manifest_hash':report['cache_manifest_hash'],'versions':versions()},'models':models}
candidate=MODEL_ROOT/'oracle-model-schema7.json'; candidate.write_text(json.dumps(artifact,separators=(',',':')),encoding='utf-8')
print(f'Validated artifact written to {candidate}. Promote it deliberately after browser parity tests.')

## Final interpretation

These are the only untouched performance estimates in v2. Do not revise features, calibration, hyperparameters, or thresholds after viewing them. Any change starts a new version with new final sessions or explicitly labels the result exploratory.